# iprPy free_energy calculation

In [1]:
# Standard library imports
import datetime
from copy import deepcopy

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-30 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('free_energy')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# free_energy calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The free_energy calculation style uses thermodynamic integration to evaluate
the absolute Helmholtz and Gibbs free energies of a solid phase by comparing
it to a reference Einstein solid. 

### Version notes

- 2022-09-20: Calculation first added to iprPy
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- The simulations used by this method are all fixed volume and therefore will not relax the system to a given pressure. Instead, it is expected that the given system is already relaxed to the target pressure of interest.


## Method and Theory

The calculation performs two different simulations: an nvt simulation to estimate Einstein solid spring constants for each atom type, and thermodynamic integrations from the selected interatomic potential to the Einstein solid model and back.  The method follows what is described in [Freitas, Asta, de Koning, Computational Materials Science 112 (2016) 333–341](https://doi.org/10.1016/j.commatsci.2015.10.050).

The Einstein solid spring constants, $k_i$, are evaluated using an nvt simulation run and measuring the mean squared displacements, $\left<\left( \Delta r_i \right)^2\right>$, averaged for each atom type $i$ and over time

$$ k_i = \frac{3 k_B T}{\left<\left( \Delta r_i \right)^2\right>} $$

The free energy integration works by first allowing the system to come to equilibrium with the target potential.  Next, "switching" steps are performed during which the interatomic potential gradually switches from the target potential to the reference potential. The system is then allowed to equilibrate using the reference potential before performing another switching stage back to the target potential.

Once the simulations are done, the measured Hamiltonian energy is integrated across the potential transformations allowing for the work of transformation to be estimated.  The work of transformation can then be added to the Gibbs free energy of the reference potential state to compute the absolute Gibbs free energy of the target potential state.

## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "free_energy.py"

# Python script created by Lucas Hale

# Standard library imports
from typing import Optional, Union

# http://www.numpy.org/
import numpy as np
import numpy.typing as npt

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj
from atomman.tools import aslist

def free_energy(lammps_command: Union[str, LAMMPSobj],
                system: am.System,
                potential: lammpspotential,
                temperature: float,
                mpi_command: Optional[str] = None,
                spring_constants: Union[float, npt.ArrayLike, None] = None,
                equilsteps: int = 25000,
                switchsteps: int = 50000,
                springsteps: int = 50000,
                pressure: unitfloat = 0.0,
                createvelocities: bool = True,
                randomseed: Optional[int] = None,
                usefiles: bool = False) -> dict:
    """
    Performs a full dynamic relax on a given system at the given temperature
    to the specified pressure state.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The system to perform the calculation on.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    temperature : float
        The temperature to run at.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    spring_constants : float, array-like object or None, optional
        The Einstein solid spring constants to assign to each atom type.  If
        None (default), then a separate simulation will estimate them using
        mean squared displacements.
    equilsteps : int, optional
        The number of equilibration timesteps at the beginning of simulations
        to ignore before evaluations.  This is used at the beginning of both
        the spring constant estimate and before each thermo switch run.
        Default value is 25000.
    switchsteps : int, optional
        The number of integration steps to perform during each of the two
        switch runs.  Default value is 50000.
    springsteps : int, optional
        The number of integration steps to perform for the spring constants
        estimation, which is only done if spring_constants is None.  Default
        value is 50000.
    pressure : float, optional
        A value of pressure to use for computing the Gibbs free energy from
        the Helmholtz free energy.  NOTE: this is not used to equilibrate the
        system during this calculation!  Default value is 0.0.
    randomseed : int or None, optional
        Random number seed used by LAMMPS.  Default is None which will select
        a random int between 1 and 900000000.
    
    Returns
    -------
    dict
        Dictionary of results consisting of keys:
        
        - **'spring_constants'** (*list*) - The Einstein spring constants
          assigned to each atom type.
        - **'work_forward'** (*float*) - The work/atom during the
          forward switching step.
        - **'work_reverse'** (*float*) - The work/atom during the
          reverse switching step.
        - **'work'** (*float*) - The reversible work/atom.
        - **'Helmholtz_reference'** (*float*) - The Helmholtz free energy/atom
          for the reference Einstein solid.
        - **'Helmholtz'** (*float*) - The Helmholtz free energy/atom.
        - **'Gibbs'** (*float*) - The Gibbs free energy/atom.
    """
    # Create a LAMMPS object if needed
    lmp = LAMMPS(lammps_command, mpi_command=mpi_command, potential=potential)

    # Convert values given with units if needed
    pressure = uc.set_in_units(pressure)

    # Set randomseed
    randomseed = am.lammp

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'F:/LAMMPS/current/bin/lmp.exe'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 23 Jun 2022 - Update 2


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial system

- __system__ is an atomman.System to use as the starting configuration.  Here, it is taken as the final configuration from the relax_dynamic calculation.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Load final configuration from relax_dynamic
system = am.load('atom_dump', '../relax_dynamic/220000.dump', symbols='Ni')
print('# of atoms in system =', system.natoms)

# of atoms in system = 4000


### 3.4. Calculation-specific parameters

- __temperature__ is the temperature to run the calculation at.
- __spring_constants__ are Einstein solid spring constants to assign to each atom type.  If None (default), then a separate simulation will estimate them using mean squared displacements.
- __equilsteps__ is the number of integration steps to perform both before estimating the spring constants and before each thermodynamic switch.  Default value is 25000.
- __switchsteps__ is the number of integration steps to perform during the thermodynamic switches. Default value is 50000.
- __springsteps__ is the number of integration steps to perform when evaluating the spring constants.  Default value is 50000.
- __pressure__ is the value of pressure to use for computing the Gibbs free energy from the Helmholtz free energy.  NOTE: this is not used to equilibrate the system!
- __createvelocities__ indicates if new atomic velocities are to be assigned to the atoms prior to any MD runs.
- __randomseed__ is a random number seed between 1 and 9000000 to use for initializing velocities and use with the langevin thermostat.  Default value of None will pick a random value.

In [9]:
temperature = 300.0
spring_constants = None
equilsteps = 25000
springsteps = 50000
switchsteps = 50000
pressure = 0.0
createvelocities = True
randomseed = None

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [10]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.free_energy.free_energy'

In [11]:
results_dict = calculation.calc(lammps_command, system, potential, temperature,
                                mpi_command = mpi_command,
                                spring_constants = spring_constants,
                                equilsteps = equilsteps,
                                switchsteps = switchsteps,
                                springsteps = springsteps,
                                pressure = pressure,
                                createvelocities = createvelocities,
                                randomseed = randomseed)
print(results_dict.keys())

dict_keys(['spring_constants', 'work_forward', 'work_reverse', 'work', 'Helmholtz_reference', 'Helmholtz', 'Gibbs'])


### 4.2. Report results

Values returned in the results_dict:

- **'spring_constants'** (*list*) - The Einstein spring constants assigned to each atom type.
- **'work_forward'** (*float*) - The work/atom during the forward switching step.
- **'work_reverse'** (*float*) - The work/atom during the reverse switching step.
- **'work'** (*float*) - The reversible work/atom.
- **'Helmholtz_reference'** (*float*) - The Helmholtz free energy/atom for the reference Einstein solid.
- **'Helmholtz'** (*float*) - The Helmholtz free energy/atom.
- **'Gibbs'** (*float*) - The Gibbs free energy/atom.

In [12]:
energy_unit = 'eV'

print('F =', uc.get_in_units(results_dict['Helmholtz'], energy_unit), f'{energy_unit}/atom')
print('G =', uc.get_in_units(results_dict['Gibbs'], energy_unit), f'{energy_unit}/atom')

F = -4.450578628408745 eV/atom
G = -4.450578628408745 eV/atom


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [13]:
calculation.clean_files()